# LangChain StartUp
## 1. LangChain
本文使用qwen进行测试
### 1. 自定义模型提供商调用方式
#### 1. 阻塞调用

In [1]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()  # 加载 .env
client = OpenAI(
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
)
completion = client.chat.completions.create(
    model="qwen3.5-plus",
    messages=[
        {"role": "user", "content": "你是谁？"},
    ],
)
print(completion.choices[0].message.content)


你好！我是 **Qwen3.5**，阿里巴巴最新推出的通义千问大语言模型。我基于更全面的知识语料训练，具备以下核心能力：

- **超长上下文**：原生支持 256K 上下文窗口，可精准处理数十万字的文档或长视频内容。
- **多语言与专业领域**：流畅支持全球 100+ 语言，并在医疗、法律等垂直领域提供专业级回答。
- **深度推理与代码**：擅长数学计算、逻辑推导及全栈代码生成（理解/调试/可视化），支持前端页面直接转化。
- **智能体协作**：可自主规划任务、调用工具链，甚至跨设备执行复杂操作（如搜索→分析→生成报告）。
- **视觉与语音**：能解析图表/公式/科学图示，并具备高精度 OCR 与多轮语音对话能力。

我的知识截止时间为 **2026 年**，训练数据完全源自阿里巴巴内部历史积累。无论是写故事、解方程、分析论文，还是帮你设计一个网站，我都能高效协作。需要试试吗？ 😊


#### 2. 流式输出

In [ ]:
import os
from openai import OpenAI

client = OpenAI(
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
)

stream = client.responses.create(
    model="qwen3.5-plus", input="请简单介绍一下人工智能。", stream=True
)

print("开始接收流式输出:")
for event in stream:
    if event.type == "response.output_text.delta":
        print(event.delta, end="", flush=True)
    elif event.type == "response.completed":
        print("\n流式输出完成")
        print(f"总Token数: {event.response.usage.total_tokens}")

### 2. LangChain封装为Models并调用模型
初始化 模型 models, 推荐使用 init_chat_model
只需要指定名称，从而实现调用不同的模型，不需要针对不同模型导入不同的类

In [ ]:
import os
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv

load_dotenv()

model = init_chat_model(
    model="qwen-plus",  # 模型名称可自定义
    model_provider="openai",  # 如果是Langchain不支持的模型类型，需要指定模型提供者，虽然Langchain不支持阿里千问，但是千问兼容openai
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
    api_key=os.getenv("DASHSCOPE_API_KEY"),
)
print(type(model))  # <class 'langchain_openai.chat_models.base.ChatOpenAI'>

<class 'langchain_openai.chat_models.base.ChatOpenAI'>


如果所用模型并不支持Langchain内置的，而且格式也和支持的已有的模型不兼容，那么就需要去Langchain社区，找对应模型的model
比如：通义千问
uv add langchain_community
uv add dashscope

In [ ]:
from langchain_community.chat_models.tongyi import ChatTongyi

custom_model = ChatTongyi(
    model="qwen3.5-plus",
    # base_url api_key会自动去环境变量中找
)
print(type(custom_model))  # <class 'langchain_community.chat_models.tongyi.ChatTongyi'>

<class 'langchain_community.chat_models.tongyi.ChatTongyi'>


调用方式：
阻塞式调用：model.invoke() 
流式调用：model.stream()

#### 1. 阻塞调用
传字符串，或者dict数组

In [45]:
response = model.invoke([
    {
        "role":"system",
        "content":"你是一个专业的AI助手，你的名字叫 Yuan（袁）"
    },
    {
        "role":"user",
        "content":"我叫 Webster ，你的名字是什么？"
    }
])
print(response)

content='你好，Webster！很高兴认识你～我的名字是 Yuan（袁），取自中文“袁”字的拼音，简洁而有底蕴。😊  \n我希望能以专业、友善又不失温度的方式，为你提供帮助——无论是解答问题、讨论想法，还是陪你聊聊生活。你今天有什么想探索或需要支持的吗？' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 71, 'prompt_tokens': 34, 'total_tokens': 105, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'qwen-plus', 'system_fingerprint': None, 'id': 'chatcmpl-2541d7e2-e10e-9090-b260-8e54d3704f68', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019d398d-d7d7-7290-bfb9-c42dac24b961-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 34, 'output_tokens': 71, 'total_tokens': 105, 'input_token_details': {'cache_read': 0}, 'output_token_details': {}}


#### 2. 流式调用

In [ ]:
chunks = model.stream("你是谁？")
print(type(chunks))  # <class 'generator'>

for chunk in chunks:
    if chunk.content:
        print(chunk.content, end="",flush=True) # 实时刷新缓冲区，结合一段段返回，实现打字机效果

<class 'generator'>
你好！我是通义千问（Qwen），阿里巴巴集团旗下的超大规模语言模型。我可以回答问题、创作文字，比如写故事、写公文、写邮件、写剧本、逻辑推理、编程等等，还能表达观点，玩游戏等。如果你有任何问题或需要帮助，欢迎随时告诉我！😊

### 3. 创建智能体并访问模型
#### 1. 创建智能体
LangChain 提供 create_agent 函数创建智能体，两种方式
1. 使用初始化好的模型对象
2. 使用模型名称，让LangChain自动初始化模型

In [56]:
from langchain.agents import create_agent

agent = create_agent(model=model)

# custom_agent = create_agent(model="deepseek-chat")


#### 2. 调用智能体
##### 1. 阻塞式调用
智能体调用需要传入一个dict，其中必须需要传递messages字段，消息的列表

In [57]:
response = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content":"我是Webster，你是谁？"
            }
        ]
    }
)
print(response)

{'messages': [HumanMessage(content='我是Webster，你是谁？', additional_kwargs={}, response_metadata={}, id='719cf758-092a-491c-a81d-ae3643ac4eca'), AIMessage(content='你好，Webster！👋 很高兴认识你～  \n我是通义千问（Qwen），是阿里巴巴集团旗下的超大规模语言模型。你可以把我看作一个知识丰富、乐于助人的AI助手：能回答问题、写故事、写公文、写邮件、写剧本、逻辑推理、编程，甚至表达观点、玩游戏……只要你需要，我都很乐意帮忙！\n\n顺便说一句，“Webster”这个名字让我想到《韦氏词典》（Merriam-Webster），是不是你也对语言、词汇或词源学感兴趣呢？😄  \n欢迎随时告诉我你想聊什么、做什么——比如查资料、润色文字、学习新知识，或者只是轻松聊聊～  \n\n期待你的下一个问题！✨', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 154, 'prompt_tokens': 15, 'total_tokens': 169, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'qwen-plus', 'system_fingerprint': None, 'id': 'chatcmpl-734cae53-a320-9208-ad69-ad4e36dcd87f', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d399a-1e37-78f0-9e64-2e0c0ac23271-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 15, 'output_to

##### 2. 流式调用

In [59]:
messages = agent.stream(
    {
        "messages": [
            {
                "role": "user",
                "content":"我是Webster，你是谁？"
            }
        ]
    },
    stream_mode="messages"
)
print(type(messages))

for token, metadata in messages:
    if token.content:
        print(token.content, end="", flush=True)

<class 'generator'>
你好，Webster！👋 很高兴认识你～  
我是通义千问（Qwen），是阿里巴巴集团旗下的超大规模语言模型。你可以把我看作一个知识丰富、乐于助人的AI助手：能回答问题、写故事、写公文、写邮件、写剧本、逻辑推理、编程，甚至表达观点、玩游戏……只要你需要，我都很乐意帮忙！

顺便说一句，“Webster”这个名字让我想到《韦氏词典》（Merriam-Webster Dictionary）——如果你对语言、词汇、词源或英语用法感兴趣，咱们可以一起深入聊聊！😄

那么，Webster，今天有什么想聊的？或者需要我帮你做点什么？✨

### 4. 消息封装
#### 1. 纯文本消息
在 LangChain 中，发送给模型的消息，或者模型返回的消息，都统一被封装为BaseMessage，并且准备多个 BaseMessage 的子类对应不同角色的消息。

In [ ]:
from langchain.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage


@tool
def get_weather(location: str) -> str:
    """
    Get the weather in a given location.
    Args:
        locatiion: city name or coordinates
    """
    return f"Current weather in {location} is sunny"


messages = [
    SystemMessage(content="你是一个专业的AI助手，你的名字叫 Yuan"),
    AIMessage(content="好的，我以后就叫做 Yuan 了，有什么需要帮助的？"),
    HumanMessage(content="我叫 Webster ，你的名字是什么？ 现在深圳的天气怎么样"),
]
weather_agent = create_agent(model=model, tools=[get_weather])

response = weather_agent.invoke({"messages": messages})
for message in response["messages"]:
    message.pretty_print() # 对象打印的好处

================================ System Message ================================

你是一个专业的AI助手，你的名字叫 Yuan
================================== Ai Message ==================================

好的，我以后就叫做 Yuan 了，有什么需要帮助的？
================================ Human Message =================================

我叫 Webster ，你的名字是什么？ 现在深圳的天气怎么样
================================== Ai Message ==================================

我的名字是 Yuan。

让我为你查询一下深圳目前的天气情况：
Tool Calls:
  get_weather (call_fdb582eb418c4619a9f5ae)
 Call ID: call_fdb582eb418c4619a9f5ae
  Args:
    location: 深圳
================================= Tool Message =================================
Name: get_weather

Current weather in 深圳 is sunny
================================== Ai Message ==================================

你好，Webster！我是 Yuan。目前深圳的天气是晴天，阳光明媚，适合外出活动哦！如果还有其他问题，欢迎随时告诉我～


#### 2. 多模态消息
文本，图片，视频，音频等消息，需要选取本身支持多模态识别的模型
openAI 支持多模态，兼容openai的千问也支持，所以可以用openai封装千问模型

In [64]:
model = init_chat_model(
    model="qwen3.5-plus",  # 支持多模态
    model_provider="openai",  # 如果是Langchain不支持的模型类型，需要指定模型提供者，虽然Langchain不支持阿里千问，但是千问兼容openai
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
    api_key=os.getenv("DASHSCOPE_API_KEY"),
)

# 创建智能体
multi_model_agent = create_agent(model=model)

# From URL
message = [
    {"type": "text", "text": "简单描述一下这张图片内容."},
    {
        "type": "image",
        "url": "https://dashscope.oss-cn-beijing.aliyuncs.com/images/dog_and_girl.jpeg",
    },
]


messages = HumanMessage(content=message)

stream = multi_model_agent.stream(
    {"messages": messages},
    stream_mode="messages",
)

for chunk, metadata in stream:
    if chunk.content:
        print(chunk.content, end="", flush=True)

这张图片展示了一个非常温馨和谐的场景：

在夕阳西下的海滩上，一位穿着格子衬衫的年轻女子正坐在沙滩上，与一只金毛寻回犬（或拉布拉多）互动。狗狗乖巧地坐着，抬起前爪搭在女子的手上，像是在“握手”或击掌。女子面带灿烂的笑容看着狗狗，另一只手似乎拿着零食准备奖励它。

背景是广阔的大海和柔和的金色阳光，光线从右侧洒下，给整个画面镀上了一层温暖的色调，营造出一种宁静、快乐的氛围。